# Smartwatch Health Analytics — Corrected ML Pipeline

**Corrected version**

This notebook fixes the `TypeError: boolean value of NA is ambiguous` error.

Key decisions:
- Categorical columns are explicitly kept as `object`
- Missing categorical values are converted to `np.nan`
- `SimpleImputer(strategy="most_frequent")` handles categorical missing values
- Numeric missing values are median-imputed
- No sequential IQR outlier deletion
- `Stress Category` is excluded from model features to avoid target leakage
- `Stress Level` remains the original integer target
- No permanent prediction or extra target column is added


In [ ]:
# In the earlier version of the smartwatch product stress report we found the below readings:

Accuracy: 0.18453333333333333
              precision    recall  f1-score   support

           1       0.26      0.32      0.29       173
           2       0.17      0.20      0.18       182
           3       0.12      0.11      0.11       189
           4       0.13      0.11      0.12       189
           5       0.16      0.14      0.15       192
           6       0.10      0.09      0.10       191
           7       0.11      0.11      0.11       191
           8       0.18      0.14      0.16       193
           9       0.22      0.24      0.23       194
          10       0.34      0.41      0.38       181

    accuracy                           0.18      1875
   macro avg       0.18      0.19      0.18      1875
weighted avg       0.18      0.18      0.18      1875

[[56 35 34 16  8 11  8  2  3  0]
 [38 36 24 18 16 14 19  7  5  5]
 [36 29 20 27 19 20 17  5 10  6]
 [28 20 23 21 25 20 22 10 14  6]
 [19 23 17 27 26 16 20 15 13 16]
 [16 16 23 16 16 18 24 25 21 16]
 [12 24  9 16 20 17 21 20 28 24]
 [ 6 11 10  9 19 23 25 27 33 30]
 [ 4 13  8 10 10 19 24 19 46 41]
 [ 4  2  4  5  6 15 16 18 36 75]]

"""Here we can see the accuracy is below 19%, and along with this precision, recall,and  f1-score are ranging between (11%-29%). Which is not 
  good enough to support the model. So now, in this workbook we will be working on all the 25 features and will go through the final analysis 
  and model building. """

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")


In [ ]:
file_path = "Smartwatch_Health_Analytics_COMPLEX_DEMO.csv"
dc = pd.read_csv(file_path)

dc.columns = dc.columns.str.strip()

print("Dataset shape:", dc.shape)
print("Rows:", len(dc))
display(dc.head())

In [ ]:
dc.shape

In [ ]:
# Define categorical columns explicitly.
# We use traditional pandas object dtype, as requested.

categorical_cols = [
    "Activity Level",
    "Stress Category",
    "Workout Intensity",
    "Sleep Quality",
    "Device Condition",
    "Stress Trigger"
]

categorical_cols = [c for c in categorical_cols if c in dc.columns]

# Clean categorical columns while retaining object dtype.
for col in categorical_cols:
    dc[col] = dc[col].astype(object)
    dc[col] = dc[col].where(pd.notna(dc[col]), np.nan)
    dc[col] = dc[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Correct known spelling/category inconsistencies.
if "Activity Level" in dc.columns:
    dc["Activity Level"] = dc["Activity Level"].replace({
        "Actve": "Active",
        "active": "Active",
        " ACTIVE": "Active",
        "Seddentary": "Sedentary",
        "Highly_Active": "Highly Active"
    })

if "Stress Trigger" in dc.columns:
    dc["Stress Trigger"] = dc["Stress Trigger"].replace({
        "Extream": "Extreme"
    })

if "Sync Status" in dc.columns:
    dc["Sync Status"] = dc["Sync Status"].replace({
        "Sync Delayed": "Delayed",
        "ESync Delayed": "Delayed"
    })

print("Categorical dtypes:")
print(dc[categorical_cols].dtypes)


In [ ]:
# Convert numerical columns safely

numeric_cols = [
    "User ID",
    "Heart Rate (BPM)",
    "Blood Oxygen Level (%)",
    "Step Count",
    "Sleep Duration (hours)",
    "Stress Level",
    "Resting Heart Rate (BPM)",
    "Heart Rate Variability (ms)",
    "Respiratory Rate (breaths/min)",
    "Active Minutes",
    "Sleep Efficiency (%)",
    "Skin Temperature (°C)",
    "Calories Burned (kcal)",
    "Sedentary Minutes",
    "Device Usage (hours)",
    "Daily Stress Events",
    "Screen Time (hours)",
    "Caffeine Intake (mg)",
    "Hydration (L)"
]

numeric_cols = [c for c in numeric_cols if c in dc.columns]

for col in numeric_cols:
    dc[col] = pd.to_numeric(dc[col], errors="coerce")

print(dc.dtypes)


In [ ]:
# Missing-value audit

missing_summary = (
    dc.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Values"] / len(dc) * 100
).round(2)

display(missing_summary[missing_summary["Missing Values"] > 0])


## Domain validation — no IQR deletion

We do **not** sequentially remove statistical outliers.

Legitimate smartwatch observations can be unusual. We only remove values that are clearly outside plausible physical/domain limits.

Missing values are preserved and handled later by the preprocessing pipeline.


In [ ]:
rules = {
    "Heart Rate (BPM)": (35, 220),
    "Resting Heart Rate (BPM)": (30, 150),
    "Sleep Duration (hours)": (0, 24),
    "Sleep Efficiency (%)": (0, 100),
    "Blood Oxygen Level (%)": (70, 100),
    "Stress Level": (1, 10),
    "Respiratory Rate (breaths/min)": (5, 40)
}

mask = pd.Series(True, index=dc.index)

for col, (low, high) in rules.items():
    if col in dc.columns:
        mask &= dc[col].between(low, high).fillna(True)

before = len(dc)
dc = dc.loc[mask].copy()

print("Rows before validation:", before)
print("Rows after validation :", len(dc))
print("Rows removed          :", before - len(dc))


## Feature engineering

The engineered variables below are used only as model inputs.

`Stress Level` remains unchanged, and `Stress Category` is excluded because it is derived from the target and can cause target leakage.


In [ ]:
# Useful domain features — created for modelling only

if {"Heart Rate (BPM)", "Resting Heart Rate (BPM)"}.issubset(dc.columns):
    dc["HR_minus_Resting"] = (
        dc["Heart Rate (BPM)"] -
        dc["Resting Heart Rate (BPM)"]
    )

if {"Heart Rate (BPM)", "Resting Heart Rate (BPM)"}.issubset(dc.columns):
    dc["HR_ratio_Resting"] = (
        dc["Heart Rate (BPM)"] /
        (dc["Resting Heart Rate (BPM)"] + 1)
    )

if "Sleep Duration (hours)" in dc.columns:
    dc["Sleep_Deficit"] = (
        8 - dc["Sleep Duration (hours)"]
    ).clip(lower=0)

if {"Active Minutes", "Sedentary Minutes"}.issubset(dc.columns):
    dc["Activity_Density"] = (
        dc["Active Minutes"] /
        (dc["Active Minutes"] + dc["Sedentary Minutes"] + 1)
    )

if {"Heart Rate Variability (ms)", "Sleep Duration (hours)",
    "Sleep Efficiency (%)"}.issubset(dc.columns):
    dc["Recovery_Index"] = (
        dc["Heart Rate Variability (ms)"] *
        dc["Sleep Duration (hours)"] *
        dc["Sleep Efficiency (%)"] / 100
    )

print("Feature engineering completed.")


In [ ]:
# Prepare target and features

target = "Stress Level"

if target not in dc.columns:
    raise ValueError("Stress Level column not found.")

# Explicitly exclude target and leakage column.
drop_cols = [target]
if "Stress Category" in dc.columns:
    drop_cols.append("Stress Category")

X = dc.drop(columns=drop_cols)
y = dc[target].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
display(y.value_counts().sort_index())


In [ ]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


In [ ]:
# Identify feature types from the actual X dataframe

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nCategorical dtypes:")
print(X[categorical_features].dtypes)


In [ ]:
# Robust preprocessing
#
# Numeric:
#   missing -> median
#
# Categorical:
#   object dtype + np.nan
#   missing -> most frequent
#   unknown categories -> ignored

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent",
        missing_values=np.nan
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## Random Forest

In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, average="macro", zero_division=0)
rf_recall = recall_score(y_test, rf_pred, average="macro", zero_division=0)
rf_f1 = f1_score(y_test, rf_pred, average="macro", zero_division=0)

print("Random Forest Results")
print("=" * 40)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))


In [ ]:
# Random Forest confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    labels=sorted(y.unique())
)

plt.title("Random Forest — Stress Level")
plt.xlabel("Predicted Stress Level")
plt.ylabel("Actual Stress Level")
plt.show()


## XGBoost

In [ ]:
# XGBoost multi-class model
# Convert labels 1–10 to 0–9 internally.

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=2,
        gamma=0.1,
        objective="multi:softmax",
        num_class=10,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train - 1)

xgb_pred = xgb_model.predict(X_test).astype(int) + 1

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred, average="macro", zero_division=0)
xgb_recall = recall_score(y_test, xgb_pred, average="macro", zero_division=0)
xgb_f1 = f1_score(y_test, xgb_pred, average="macro", zero_division=0)

print("XGBoost Results")
print("=" * 40)
print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred, zero_division=0))


In [ ]:
# XGBoost confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_pred,
    labels=sorted(y.unique())
)

plt.title("XGBoost — Stress Level")
plt.xlabel("Predicted Stress Level")
plt.ylabel("Actual Stress Level")
plt.show()


In [ ]:
# Compare models

comparison = pd.DataFrame({
    "Model": [
        "Improved Random Forest",
        "Improved XGBoost"
    ],
    "Accuracy": [
        rf_accuracy,
        xgb_accuracy
    ],
    "Macro Precision": [
        rf_precision,
        xgb_precision
    ],
    "Macro Recall": [
        rf_recall,
        xgb_recall
    ],
    "Macro F1": [
        rf_f1,
        xgb_f1
    ]
})

display(comparison.round(4))


In [ ]:
# Final data-type verification

print("Final categorical dtypes:")
display(dc[categorical_cols].dtypes)

print("\nFinal target dtype:")
print(dc["Stress Level"].dtype)

print("\nFinal shape:")
print(dc.shape)


## Why `object` instead of `string[python]`?

The categorical columns are deliberately stored as traditional pandas `object` columns.

The earlier `string[python]` dtype came from explicitly using:

```python
.astype("string")
```

That dtype uses pandas `pd.NA` for missing text values. In the previous pipeline, this interacted badly with the preprocessing/imputation path and produced:

`TypeError: boolean value of NA is ambiguous`

The corrected pipeline uses:

```python
.astype(object)
```

and converts missing categorical values to:

```python
np.nan
```

This is compatible with the `SimpleImputer` configuration used here.


# Production ML Pipeline & Deployment

This section converts the preprocessing + model steps into a **single reproducible pipeline**.

The important production principle is:

> The same preprocessing used during training must automatically be applied when new smartwatch data is submitted.

This prevents training/production preprocessing mismatch.


In [ ]:
# Select the final model automatically
# XGBoost is selected here as the deployment candidate.
# If Random Forest performs better after evaluation, change this to rf_model.

final_model = xgb_model

print("Deployment model:", type(final_model.named_steps["model"]).__name__)


## Save the complete pipeline

The saved object contains:

1. Missing-value handling
2. Categorical encoding
3. Feature transformation
4. Trained machine-learning model

Therefore, a new raw dataframe can be passed directly to the saved pipeline.


In [ ]:
import joblib

model_path = "smartwatch_stress_pipeline.pkl"

joblib.dump(final_model, model_path)

print(f"Pipeline saved successfully: {model_path}")


In [ ]:
# Load the saved production pipeline

loaded_pipeline = joblib.load("smartwatch_stress_pipeline.pkl")

print("Pipeline loaded successfully.")
print(loaded_pipeline)


## Prediction function

This function accepts **raw feature values** and lets the saved pipeline perform the required preprocessing automatically.

No manually encoded categorical columns are required.


In [ ]:
def predict_stress(input_data, pipeline=loaded_pipeline):
    """
    input_data: pandas DataFrame containing the same input feature columns
                used during model training.

    Returns:
        Predicted Stress Level (1–10)
    """
    prediction = pipeline.predict(input_data)
    return prediction


# Example: use one record from the test set.
sample_input = X_test.iloc[[0]].copy()

prediction = predict_stress(sample_input)

print("Predicted Stress Level:", int(prediction[0]))
print("Actual Stress Level   :", int(y_test.iloc[0]))


## Production input validation

Before making a prediction, verify that the incoming data contains the expected model features.


In [ ]:
expected_features = X.columns.tolist()

def validate_input(input_data):
    missing_features = [
        col for col in expected_features
        if col not in input_data.columns
    ]

    extra_features = [
        col for col in input_data.columns
        if col not in expected_features
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    # Extra columns are ignored rather than silently changing the model input.
    return input_data[expected_features].copy()


validated_sample = validate_input(sample_input)

print("Input validation successful.")
display(validated_sample)


# Optional REST API Deployment with FastAPI

A simple deployment architecture is:

```text
User / Application
        |
        v
   FastAPI Endpoint
        |
        v
Input Validation
        |
        v
Saved ML Pipeline
        |
        v
Stress Level Prediction
        |
        v
      JSON
```

The API does **not** need separate preprocessing code because the preprocessing is already inside the saved pipeline.


In [ ]:
# Save this code as app.py for API deployment.

api_code = r'''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Smartwatch Stress Prediction API")

model = joblib.load("smartwatch_stress_pipeline.pkl")


class PredictionRequest(BaseModel):
    data: dict


@app.get("/")
def home():
    return {"message": "Smartwatch Stress Prediction API is running"}


@app.post("/predict")
def predict(request: PredictionRequest):
    try:
        input_df = pd.DataFrame([request.data])
        prediction = model.predict(input_df)

        return {
            "predicted_stress_level": int(prediction[0])
        }

    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(api_code)

print("Created app.py")


## Run the API locally

Install:

```bash
pip install fastapi uvicorn joblib pandas scikit-learn xgboost
```

Run:

```bash
uvicorn app:app --reload
```

Then open:

```text
http://127.0.0.1:8000/docs
```

The Swagger interface can be used to test the prediction endpoint.


# Deployment options

For your project presentation, explain deployment in three layers:

### 1. Model artifact
`smartwatch_stress_pipeline.pkl`

Contains preprocessing + trained model.

### 2. API layer
`FastAPI`

Provides a `/predict` endpoint.

### 3. Application layer
A web/mobile/dashboard application can send smartwatch measurements to the API and receive the predicted stress level.

```text
Smartwatch Data
      ↓
Application
      ↓
FastAPI
      ↓
ML Pipeline
      ↓
Stress Level 1–10
```

### Important production note

The deployed pipeline should use the **same feature names and feature definitions used during training**. Do not manually recreate the encoder or imputer in the application.


# Project Pipeline — Final Presentation Version

```text
RAW SMARTWATCH DATA
        |
        v
DATA QUALITY CHECK
        |
        v
DATA CLEANING
        |
        +----------------------+
        |                      |
        v                      v
Numerical Features      Categorical Features
        |                      |
        v                      v
Median Imputation       Mode Imputation
        |                      |
        |                One-Hot Encoding
        +----------+-----------+
                   |
                   v
          FEATURE ENGINEERING
                   |
                   v
            TRAIN / TEST SPLIT
                   |
             +-----+-----+
             |           |
             v           v
       Random Forest  XGBoost
             |           |
             +-----+-----+
                   |
                   v
              EVALUATION
                   |
                   v
             MODEL SELECTION
                   |
                   v
       COMPLETE ML PIPELINE
                   |
                   v
              JOBLIB FILE
                   |
                   v
             FASTAPI / API
                   |
                   v
        PREDICT STRESS LEVEL
```
